# N6 - External Consensus

공개 상위 제출물 6종 + 우리 N5(HGB+TE 10fold×7seed)의 **행별 하드보팅**.
- 동률 tie-break: ①N5 ②public LB 높은 소스 순
- ⚠️ 정직 라벨: 이 카드는 CV 검증 불가(공개 산출물 재활용, public LB 추종 성격).
  최종 2개 선택에서 '공격 카드'로만 사용 — '방어 카드'는 정직 CV의 N5/N7.
- 소스는 전부 kernel source로 부착 (재현 가능)

In [ ]:
import numpy as np, pandas as pd, glob, os, itertools

# 대회 데이터(sample_submission) & 소스 제출물 자동 탐색
comp = sorted(glob.glob('/kaggle/input/**/sample_submission.csv', recursive=True))
assert comp, 'competition data not mounted'
ss = pd.read_csv(comp[0])
CLASSES = ['at-risk','unhealthy','fit']

print('input tree:', {d: os.listdir(f'/kaggle/input/{d}')[:5] for d in os.listdir('/kaggle/input')})
subs = {}
for p in sorted(glob.glob('/kaggle/input/**/submission.csv', recursive=True)):
    d = os.path.basename(os.path.dirname(p))          # 슬러그 = 마지막 폴더명
    if 'playground' in d or 'competitions' in d:
        continue
    subs[d] = pd.read_csv(p)
print('sources found:')
for k, v in subs.items(): print(' ', k, v.shape)
assert len(subs) >= 5, f'source shortage: {list(subs)}'

In [ ]:
# tie-break 우선순위: N5 -> LB 내림차순 (디렉토리명 매칭)
PRIORITY = ['n5-hgb', 'external-consensus', 'external-ensemble',
            'hearth', 'post-processing', 'realmlp', 'eda-ensemble']
def prio(name):
    for i, k in enumerate(PRIORITY):
        if k in name: return i
    return 99
order = sorted(subs, key=prio)
print('tie-break order:', order)

P = {}
for name, df in subs.items():
    assert len(df) == len(ss), f'{name} row mismatch'
    df = df.set_index('id').loc[ss['id']].reset_index()
    assert set(df['health_condition'].unique()) <= set(CLASSES), name
    P[name] = df['health_condition'].to_numpy()
D = pd.DataFrame(P)

print('\n=== pairwise agreement (%) ===')
for a, b in itertools.combinations(order, 2):
    print(f'{a[:24]:24s} vs {b[:24]:24s} {(D[a]==D[b]).mean()*100:.2f}')

In [ ]:
cls_idx = {c: i for i, c in enumerate(CLASSES)}
votes = np.zeros((len(D), 3), dtype=int)
for n in D.columns:
    votes[np.arange(len(D)), D[n].map(cls_idx).to_numpy()] += 1
maxv = votes.max(1)
final = np.empty(len(D), dtype=object)
ties = 0
for i in range(len(D)):
    winners = [CLASSES[k] for k in range(3) if votes[i, k] == maxv[i]]
    if len(winners) == 1:
        final[i] = winners[0]
    else:
        ties += 1
        for src in order:
            if D[src].iat[i] in winners:
                final[i] = D[src].iat[i]; break

our = [n for n in D.columns if 'n5-hgb' in n]
if our:
    ch = (final != D[our[0]].to_numpy()).sum()
    print(f'동률 {ties:,}행 | N5 대비 변경 {ch:,}행 ({ch/len(D)*100:.2f}%)')
sub = pd.DataFrame({'id': ss['id'], 'health_condition': final})
sub.to_csv('submission.csv', index=False)
assert len(sub) == len(ss) and sub['health_condition'].isna().sum() == 0
print('submission.csv | sanity OK |', sub['health_condition'].value_counts().to_dict())